# Неделя 1. Введение в обучение с подкреплением

Курс Reinforcement Learning. Вводная лекция.

**Цели лекции**

1. Выделить основные отличительные черты Reinforcement Learning.
2. Понимать области применения Reinforcement Learning.
3. Разобраться, что отличает Reinforcement Learning от других областей машинного обучения.

**План на сегодня**

1. Как устроен курс
2. Учимся методом проб и ошибок: примеры из жизни
3. Откуда взялся RL: психология и теория управления
4. Что такое RL: агент, среда, награда
5. Основные термины: состояние, действие, политика, эпизод, суммарная награда
6. Живое демо: агент в среде Gymnasium
7. Чем RL отличается от остального машинного обучения
8. Главная дилемма: исследовать или использовать
9. Где применяется RL
10. Карта курса, итоги, литература

## 0. Как устроен курс

* **15 недель**, одно занятие в неделю: лекция + семинар. Каждая неделя — папка `NN-topic-name/` в репозитории с тремя ноутбуками: `lecture/`, `seminar/`, `homework/`.
* **Домашние задания** почти каждую неделю, в `.ipynb`. Часть проверок — через `assert`, так что можно проверить себя до сдачи.
* **Оценка** (черновик, обсуждаем сегодня): 60% домашние задания, 30% итоговый проект, 10% активность на семинарах.
* **Итоговый проект**: своя реализация RL-агента или мини-исследование на среде по выбору. Темы появятся к 8-й неделе.
* **Инструменты**: Python 3.10+, [Gymnasium](https://gymnasium.farama.org/) (среды), [PyTorch](https://pytorch.org/) (нейросети), Jupyter.
  Библиотеки [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) и [CleanRL](https://github.com/vwxyzjn/cleanrl) читаем как справочник, но алгоритмы в домашках пишем сами.
* **Что нужно уметь на входе**: линейная алгебра, теория вероятностей, градиентный спуск, Python с numpy. Нейросети и PyTorch — желательно, но необходимый минимум разберём на мини-семинаре `../seminar/pytorch_intro.ipynb`.

Установка окружения:

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

## 1. Учимся методом проб и ошибок

Прежде чем давать определения, посмотрим на то, что мы и так умеем.

* **Ребёнок учится ходить.** Никто не объясняет ему, под каким углом сгибать колено. Он пробует, падает, пробует иначе. Постепенно падений становится меньше.
* **Велосипед.** Можно прочитать учебник по физике гироскопа, но кататься от этого не научишься. Учатся только садясь и падая. И, что важно, **словами объяснить, как держать равновесие, не может даже тот, кто умеет**.
* **Дрессировка собаки.** Собака сделала что-то похожее на «сидеть» — получила лакомство. Сделала не то — не получила. Никто не показывает собаке «правильный ответ», есть только «получилось / не получилось».
* **Новая настольная игра без чтения правил.** Первые партии проигрываем всем, но замечаем: вот такие ходы обычно ведут к победе, а такие — к поражению. Через десяток партий начинаем выигрывать.
* **Видеоигры.** Новый уровень в Mario или Dark Souls проходят не с первой попытки. Каждая смерть — информация: «здесь прыгать раньше», «этого противника обходить слева».
* **Шахматы, го, покер.** Пожертвовали фигуру — сейчас стало хуже, но через десять ходов выиграли партию. Хороший ход не обязан давать выгоду немедленно.

Во всех примерах одно и то же:

1. есть **кто-то, кто действует** и **что-то, что отвечает** на действия;
2. **правильного ответа никто не сообщает**, есть только последствия: приятные или неприятные;
3. последствия могут прийти **с задержкой**;
4. поведение **меняется от опыта**: то, что приводило к хорошему, повторяем чаще.

Обучение с подкреплением — это попытка описать этот механизм математически и заставить его работать в компьютере.

## 2. Откуда взялся RL: психология

Само слово *reinforcement* («подкрепление») пришло не из математики, а из психологии. Область начиналась с вопроса «как животные учатся?».

* **Иван Павлов, 1900-е.** Условный рефлекс: если звонок раз за разом предшествует еде, собака начинает выделять слюну на звонок. Обучение как связывание **стимула** и **реакции**. Здесь животное ещё ничего не выбирает, оно только предсказывает.
* **Эдвард Торндайк, 1898–1911.** Кошек сажали в «проблемный ящик»: чтобы выйти и получить еду, нужно потянуть за верёвку или нажать на рычаг. Сначала кошка мечется случайно, потом всё быстрее находит нужное действие. Торндайк сформулировал **закон эффекта** (law of effect): действия, за которыми следует удовлетворение, в той же ситуации повторяются чаще; действия с неприятными последствиями — реже. Это почти дословно определение RL.
* **Беррес Скиннер, 1930–50-е.** «Ящик Скиннера»: крыса нажимает на рычаг и получает еду. Скиннер ввёл термин **подкрепление** (reinforcement) — всё, что увеличивает частоту поведения, — и **оперантное обусловливание**: организм не просто реагирует, а *действует* и учится на последствиях своих действий. Изучал расписания подкрепления: что будет, если награда приходит не каждый раз, а случайно (спойлер: поведение закрепляется даже сильнее, так работают игровые автоматы).

Перевод на язык RL:

| Психология | RL |
|---|---|
| организм (животное) | агент |
| ситуация, стимул | состояние |
| реакция, поведение | действие |
| подкрепление | награда |
| привычка, выученное поведение | политика |
| закон эффекта | обновление политики по награде |

Первый итог лекции: **Reinforcement Learning зарождался в психологии**, и его словарь до сих пор оттуда.

## 3. Вторая ветка: теория управления, и как две линии сошлись

Параллельно и независимо математики решали задачу «как управлять системой, чтобы через много шагов было хорошо».

* **Ричард Беллман, 1950-е.** Динамическое программирование и марковские процессы принятия решений: как найти лучшее поведение, если известно, как устроена система. Отсюда идея **ценности состояния** («насколько хорошо здесь находиться») и знаменитые уравнения Беллмана, до которых мы дойдём на неделе 2. Пока запомним имя.
* **Марвин Минский, 1954–1961.** Собрал одну из первых обучающихся машин SNARC (сеть из 40 «нейронов» на лампах, учившаяся проходить лабиринт) и сформулировал проблему **credit assignment**: если награда пришла в конце, какое из ста сделанных действий её заслужило?
* **Гарри Клопф, Ричард Саттон, Эндрю Барто, 1970–80-е.** Клопф настаивал, что обучение с учителем (по правильным ответам) и обучение по подкреплению — принципиально разные вещи, и второе незаслуженно забыто. Его студенты Саттон и Барто соединили психологическую идею проб и ошибок с математикой Беллмана: **temporal-difference learning** (1988), actor-critic.
* **1989–1992.** Крис Уоткинс придумал Q-learning; Джеральд Тезауро обучил нейросеть **TD-Gammon** играть в нарды на уровне чемпионов мира. Первый громкий успех.
* **1998.** Учебник Саттона и Барто *Reinforcement Learning: An Introduction* — область оформилась окончательно.

![origins](../../assets/rl_origins.png)

### Современный этап

Долгое время RL работал только на задачах с небольшим числом состояний. Всё изменилось, когда к нему подключили глубокие нейросети:

![timeline](../../assets/rl_timeline.png)

К каждому пункту вернёмся в разделе 8, а пока главное: с 2013 года RL перестал быть академической игрушкой и стал инструментом, которым обучают роботов, играют в го и дообучают ChatGPT.

## 4. Что такое RL: агент, среда, награда

**Reinforcement Learning (RL)** — раздел машинного обучения, в котором **агент** учится принимать решения, взаимодействуя со **средой**, чтобы получить как можно больше **награды** в сумме.

![loop](../../assets/agent_env_loop.png)

Три действующих лица:

* **Агент** (agent) — тот, кто принимает решения: собака, игрок, программа, управляющая роботом. Это единственное, что мы обучаем.
* **Среда** (environment) — всё остальное: физический мир, игровое поле, биржа, другой игрок. Среда отвечает на действия агента и не обязана быть «дружелюбной» или даже предсказуемой.
* **Награда** (reward) — число, которое среда выдаёт агенту после каждого шага. Лакомство для собаки, очки в игре, минус за падение. Награда — единственный сигнал о том, хорошо агент действует или плохо.

Взаимодействие идёт по шагам. На каждом шаге $t$:

1. агент видит текущее **состояние** среды $s_t$;
2. выбирает **действие** $a_t$;
3. среда переходит в новое состояние $s_{t+1}$ и выдаёт **награду** $r_{t+1}$;
4. повторить.

Всё, что делает любой алгоритм RL, — так или иначе меняет правило выбора действия на шаге 2, глядя на то, что случилось на шаге 3.

## 5. Основные термины

Зафиксируем словарь на трёх примерах: дрессировка собаки, задача CartPole (тележка с шестом, её увидим в демо) и шахматы.

| Термин | Что это | Собака | CartPole | Шахматы |
|---|---|---|---|---|
| **Состояние** (state), $s$ | всё, что агент знает о ситуации в данный момент | поза, голос хозяина, есть ли еда в руке | положение и скорость тележки, угол и скорость шеста | расположение фигур на доске |
| **Действие** (action), $a$ | что агент может сделать | сесть, лечь, гавкнуть, убежать | толкнуть тележку влево или вправо | любой допустимый ход |
| **Награда** (reward), $r$ | число, оценка последствий одного шага | +1 лакомство, 0 ничего | +1 за каждый шаг, пока шест стоит | +1 победа, −1 поражение, 0 иначе |
| **Политика** (policy), $\pi$ | правило выбора действия по состоянию | привычки собаки | «если шест падает вправо — толкай вправо» | стиль игры шахматиста |
| **Эпизод** (episode) | одна попытка от начала до конца | одно занятие | от старта до падения шеста | одна партия |
| **Суммарная награда** (return), $G$ | сумма наград за эпизод | сколько лакомств за занятие | сколько шагов простоял шест | выиграна ли партия |

Два уточнения, которые часто путают новички.

**Состояние и наблюдение.** Агент не всегда видит среду целиком. В покере вы знаете свои карты, но не карты соперника; робот видит только то, что попало в камеру. То, что агент реально видит, называют **наблюдением** (observation). В этом курсе до недели 13 будем считать, что наблюдение и есть состояние.

**Награда и цель.** Награда приходит на каждом шаге, а цель агента — не одна награда, а их **сумма за весь эпизод**. Про это следующий раздел.

### Политика: что именно мы обучаем

**Политика** — это правило «в такой ситуации делаю так». Это единственное, что меняется в процессе обучения, поэтому термин центральный.

Политика может быть:

* **случайной**: в любой ситуации кидаем монетку. С неё обычно начинают, потому что ничего другого агент ещё не знает;
* **написанной руками** (эвристикой): «если шест падает вправо, толкай вправо». Хороша, когда задача простая и человек понимает её физику;
* **выученной**: таблица «состояние → действие» или нейросеть, параметры которой подобраны по опыту. Всё содержание курса — про то, как такую политику получить.

Политика может быть детерминированной («в состоянии $s$ всегда делай $a$») или случайной: задавать **вероятности** действий. Обозначение $\pi(a \mid s)$ читается как «вероятность выбрать действие $a$, находясь в состоянии $s$». Случайность в политике полезна: она позволяет пробовать разное, к этому вернёмся в разделе 7.

### Награда и суммарная награда: агент смотрит вперёд

Агент максимизирует не награду за ближайший шаг, а **суммарную награду** (return) до конца эпизода:

$$
G = r_1 + r_2 + r_3 + \ldots + r_T .
$$

Поэтому хороший агент умеет **терпеть**:

* шахматист жертвует ферзя (награда сейчас плохая), чтобы поставить мат через пять ходов;
* в Mario иногда нужно вернуться назад, чтобы взять гриб и пройти дальше;
* студент учится четыре года ради зарплаты после диплома.

Есть нюанс: если эпизод может длиться бесконечно, сумма расходится, а награда через тысячу шагов вряд ли должна волновать агента так же, как награда завтра. Поэтому обычно будущие награды **дисконтируют**: умножают награду через $k$ шагов на $\gamma^k$, где $\gamma$ — число чуть меньше единицы (например, 0.99). Чем меньше $\gamma$, тем «нетерпеливее» агент. Посмотрим, как быстро убывает вес будущего.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ks = np.arange(0, 200)
for gamma in [0.5, 0.9, 0.99]:
    plt.plot(ks, gamma ** ks, label=f"γ = {gamma}: агент «видит» примерно {1/(1-gamma):.0f} шагов вперёд")
plt.xlabel("k (через сколько шагов придёт награда)")
plt.ylabel("вес награды γ^k")
plt.title("Дисконтирование: насколько агенту важно будущее")
plt.legend()
plt.show()

### Награда — это формулировка задачи, и её легко испортить

Агент оптимизирует ровно то, что написано в награде, а не то, что вы имели в виду.

* **Собака**: хотели научить «сидеть», а давали лакомство каждый раз, когда она смотрела на руку. Научили смотреть на руку.
* **CoastRunners** ([разбор OpenAI](https://openai.com/index/faulty-reward-functions/)): агента-лодку обучали на игровые очки. Вместо прохождения трассы он нашёл лагуну, где можно бесконечно крутиться по кругу, собирать бонусы и врезаться в стены. Очков больше, чем у любого честного игрока.
* **Пылесос**, которому платят за собранную пыль: оптимальная стратегия — высыпать пыль обратно и собрать снова.

Это называется **reward hacking**, и это не курьёз, а повседневная проблема при работе с RL. Обзор с десятками примеров есть у [Lilian Weng](https://lilianweng.github.io/posts/2024-11-28-reward-hacking/).

Практическое правило: награда должна описывать *что* нужно получить, а не *как* это делать.

## 6. Живое демо: агент в среде Gymnasium

[Gymnasium](https://gymnasium.farama.org/) — стандартный интерфейс к средам: `env.reset()` возвращает первое наблюдение, `env.step(action)` — следующее наблюдение, награду и флаги окончания эпизода. Дальше в курсе все среды будут выглядеть так.

Возьмём классику: **CartPole**. Тележка едет по рельсу, на ней шест; нужно двигать тележку влево или вправо так, чтобы шест не упал. Наблюдение — 4 числа (положение и скорость тележки, угол и угловая скорость шеста), действий два. Награда +1 за каждый шаг, пока шест стоит; эпизод обрывается, когда шест отклонился больше 12° или тележка уехала за край.

In [ ]:
import gymnasium as gym

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
print("observation_space:", env.observation_space)
print("action_space:     ", env.action_space)
print("первое наблюдение:", obs)

obs, reward, terminated, truncated, info = env.step(1)  # 1 = толкнуть вправо
print("после шага:        ", obs, "reward =", reward, "done =", terminated or truncated)

In [ ]:
# Посмотрим, как выглядит эпизод со случайной политикой: сохраним кадры.
env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = env.reset(seed=1)
frames = []
for t in range(60):
    frames.append(env.render())
    obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
    if terminated or truncated:
        break
env.close()

idx = np.linspace(0, len(frames) - 1, 6).astype(int)
fig, axes = plt.subplots(1, 6, figsize=(16, 2.6))
for ax, i in zip(axes, idx):
    ax.imshow(frames[i])
    ax.set_title(f"t = {i}")
    ax.axis("off")
plt.suptitle(f"Случайная политика: шест падает за {len(frames)} шагов")
plt.show()

Случайная политика держит шест примерно 20 шагов. Напишем политику **руками**: если шест падает вправо (угловая скорость положительная), толкаем тележку вправо, и наоборот. Это тоже политика, просто не выученная, а придуманная человеком.

In [ ]:
def random_policy(obs):
    return np.random.randint(2)

def heuristic_policy(obs):
    x, x_dot, theta, theta_dot = obs
    # толкаем тележку в ту сторону, куда сейчас падает шест
    return int(theta_dot > 0)

def run_episodes(policy, n_episodes=50, seed=0):
    env = gym.make("CartPole-v1")
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total = 0
        while True:
            obs, r, terminated, truncated, _ = env.step(policy(obs))
            total += r
            if terminated or truncated:
                break
        returns.append(total)
    env.close()
    return np.array(returns)

np.random.seed(0)
for name, policy in [("случайная", random_policy), ("эвристика", heuristic_policy)]:
    G = run_episodes(policy)
    print(f"{name:10s}: средняя суммарная награда {G.mean():6.1f} ± {G.std():5.1f} (максимум 500)")

Эвристика примерно в 10 раз лучше случайной, но:

* её придумал человек, зная физику задачи;
* она не идеальна: до 500 шагов не дотягивает, тележка постепенно уезжает за край (на семинаре попробуете улучшить правило, добавив угол шеста и положение тележки);
* для шахмат, StarCraft или робота с 20 суставами такую эвристику руками не напишешь.

**Задача RL** — получить политику не хуже (а обычно лучше) эвристики, **не зная** устройства среды, только из опыта взаимодействия. На неделе 5 обучим на CartPole нейросеть, которая стабильно держит 500 шагов.

Если установлен `box2d`, можно посмотреть на среду посложнее: LunarLander, где нужно мягко посадить модуль на площадку.

In [ ]:
try:
    env = gym.make("LunarLander-v3", render_mode="rgb_array")
    obs, _ = env.reset(seed=0)
    frames = []
    for t in range(120):
        frames.append(env.render())
        obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
        if terminated or truncated:
            break
    env.close()
    idx = np.linspace(0, len(frames) - 1, 4).astype(int)
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for ax, i in zip(axes, idx):
        ax.imshow(frames[i]); ax.set_title(f"t = {i}"); ax.axis("off")
    plt.suptitle("LunarLander, случайная политика: наблюдение из 8 чисел, 4 действия (двигатели)")
    plt.show()
except Exception as e:  # box2d не установлен или не собрался
    print("LunarLander недоступен:", type(e).__name__, "-", str(e)[:120])
    print("Гифку можно посмотреть в документации: https://gymnasium.farama.org/environments/box2d/lunar_lander/")

## 7. Чем RL отличается от остального машинного обучения

Машинное обучение обычно делят на три группы. RL — отдельная группа, а не «ещё один вид supervised learning».

![paradigms](../../assets/ml_paradigms.png)

| | Supervised Learning | Unsupervised Learning | Reinforcement Learning |
|---|---|---|---|
| Что дано | размеченные пары (x, y) | неразмеченные x | среда, с которой можно взаимодействовать |
| Обратная связь | правильный ответ сразу | нет обратной связи | награда: число, часто с задержкой |
| Откуда данные | собраны заранее, фиксированы | собраны заранее, фиксированы | порождает сам агент своими действиями |
| Цель | предсказывать y | найти структуру в данных | максимизировать суммарную награду |
| Пример | «кошка или собака на фото?» | «на какие группы делятся клиенты?» | «как пройти уровень?» |

### Четыре отличительные черты RL

1. **Нет правильного ответа, есть оценка.** В supervised learning учитель говорит: «здесь должно быть 7». В RL никто не говорит «здесь нужно было повернуть налево»; есть только число, и оно не объясняет, что именно было хорошо или плохо.

2. **Отложенная награда и credit assignment.** Награда может прийти через 200 ходов после решающего действия. Какое из 200 действий было решающим? Награда одна на всех, и распределить её по действиям — отдельная задача.

3. **Данные порождает сам агент.** В supervised learning датасет собран заранее. В RL то, что агент увидит, зависит от того, что он делал: плохая политика видит только плохие состояния. Данные не независимы и не одинаково распределены, распределение меняется по ходу обучения. Привычные гарантии статистики ломаются.

4. **Нужно исследовать.** Чтобы найти лучшее поведение, нужно пробовать новое, а пробовать новое в среднем невыгодно. В supervised learning такой проблемы нет вообще. Это настолько важно, что заслуживает отдельного раздела.

Второй итог лекции: **Reinforcement Learning можно выделить в отдельную группу алгоритмов машинного обучения**. Общее с остальным ML — статистика и нейросети как инструмент; отличается сама постановка задачи.

## 8. Главная дилемма: исследовать или использовать

**Exploration vs exploitation.** Представьте, что вы недавно переехали в новый район.

* **Обед.** Рядом десять кафе. В трёх вы уже были: одно хорошее, два так себе. Идти в хорошее (наверняка нормально пообедаете) или попробовать одно из семи незнакомых (может быть лучше, а может хуже)?
* **Дорога на работу.** Знакомый маршрут занимает 40 минут. Навигатор предлагает другой, незнакомый. Возможно, он быстрее. Возможно, там пробка.
* **Игры.** Играть в любимую игру, где вы точно получите удовольствие, или попробовать новую?
* **Настолка.** Разыгрывать проверенную стратегию, которая приносит второе место, или рискнуть новой, которая может дать первое, а может последнее?
* **Продукт.** Показывать пользователям баннер, который уже даёт клики, или тестировать новые варианты (это в точности A/B-тестирование)?

Во всех случаях выбор один и тот же:

* **Exploitation** (использование) — делать то, что по текущим знаниям лучше всего. Даёт хороший результат *сейчас*.
* **Exploration** (исследование) — пробовать что-то другое, чтобы узнать больше. Стоит денег или времени *сейчас*, но может окупиться *потом*.

Делать только первое — значит рисковать навсегда застрять на посредственном варианте, просто потому что лучший ни разу не попробовали. Делать только второе — значит никогда не пользоваться тем, что узнали. Любой RL-агент должен как-то балансировать между ними, и мы увидим, что даже простейшее решение работает удивительно неплохо.

Проверим на «кафе».

In [ ]:
# Три кафе. Каждый обед в кафе k приносит удовольствие от 0 до 10 (среднее true_means[k] плюс шум).
# Агент не знает средних и оценивает их по своему опыту.
rng = np.random.default_rng(0)
true_means = np.array([5.0, 5.5, 7.0])   # третье кафе на самом деле лучшее, но по паре обедов этого не понять
n_days = 200


def simulate(explore_prob):
    # explore_prob: доля дней, когда идём в случайное кафе, а не в лучшее по текущей оценке
    estimate = np.zeros(3)     # текущая оценка каждого кафе
    visits = np.zeros(3)
    total = 0.0
    for day in range(n_days):
        if rng.random() < explore_prob or visits.min() == 0:
            k = rng.integers(3)                    # пробуем что-то новое
        else:
            k = int(np.argmax(estimate))           # идём в лучшее по опыту
        reward = np.clip(true_means[k] + rng.normal(0, 3.0), 0, 10)   # оценка одного обеда сильно шумит
        visits[k] += 1
        estimate[k] += (reward - estimate[k]) / visits[k]   # среднее по посещениям
        total += reward
    return total / n_days, int(np.argmax(estimate))


n_runs = 200
for p in [0.0, 0.05, 0.1, 0.3, 1.0]:
    results = [simulate(p) for _ in range(n_runs)]
    mean_reward = np.mean([r for r, _ in results])
    found_best = np.mean([best == 2 for _, best in results])
    print(f"исследуем в {p:4.0%} дней: среднее удовольствие {mean_reward:.2f}, "
          f"нашли лучшее кафе в {found_best:4.0%} запусков")

Что видно из таблицы:

* **Чисто жадный агент** (0% исследования) после первых трёх обедов навсегда ходит в то кафе, которое случайно понравилось больше. Примерно в каждом четвёртом запуске это не лучшее кафе, и агент об этом никогда не узнает: он туда больше не ходит.
* **Небольшая доля исследования** (5–10%) почти всегда находит лучшее кафе, и средний результат выше, хотя каждый десятый обед агент «тратит» на проверку.
* **Только исследование** (100%) тоже находит лучшее кафе, но не пользуется знанием: средний результат — просто среднее по трём кафе.

Разница в среднем удовольствии кажется небольшой, но она накапливается: за 200 дней это десятки «потерянных» хороших обедов, а если бы кафе было не три, а тридцать, жадный агент проигрывал бы гораздо сильнее.

Эта задача без состояний называется **многоруким бандитом** (по аналогии с игровыми автоматами: несколько рычагов, у каждого своя неизвестная выплата). На неделе 2 разберём её как следует: там есть алгоритмы гораздо умнее «иногда идём в случайное кафе», и красивая теория о том, сколько исследования нужно. Сегодня достаточно понять саму дилемму.

## 9. Где применяется RL

Общий рецепт: как только у задачи есть **среда**, с которой можно взаимодействовать (или её симулятор), и **числовая цель**, RL позволяет не программировать поведение вручную, а выучить его. Ниже области, где это сработало. У каждой есть ссылка на демо или статью: посмотрите видео, это лучшая мотивация.

### Игры: от нард до StarCraft

| Год | Система | Что произошло | Посмотреть |
|---|---|---|---|
| 1992 | **TD-Gammon** (Tesauro) | Нейросеть + TD-обучение играет в нарды на уровне чемпионов мира. Первый большой успех RL. | [статья](https://dl.acm.org/doi/10.1145/203330.203343) |
| 2013–2015 | **DQN** (DeepMind) | Одна и та же сеть учится играть в 49 игр Atari, глядя только на пиксели и счёт. | [видео Breakout](https://www.youtube.com/watch?v=TmPfTpjtdgg), [Nature](https://www.nature.com/articles/nature14236) |
| 2016 | **AlphaGo** | Победа над Ли Седолем в го. За год до этого считалось, что до этого ещё десятилетие. | [фильм AlphaGo](https://www.youtube.com/watch?v=WXuK6gekU1Y), [страница проекта](https://deepmind.google/research/breakthroughs/alphago/) |
| 2017 | **AlphaZero** | Тот же алгоритм, обучаясь только игрой с самим собой, осваивает шахматы, сёги и го с нуля. | [блог DeepMind](https://deepmind.google/discover/blog/alphazero-shedding-new-light-on-chess-shogi-and-go/) |
| 2019 | **OpenAI Five**, **AlphaStar** | Победы над профессионалами в Dota 2 и StarCraft II: длинные горизонты, частичная наблюдаемость, командная игра. | [OpenAI Five](https://openai.com/index/openai-five/), [AlphaStar](https://deepmind.google/discover/blog/alphastar-mastering-the-real-time-strategy-game-starcraft-ii/) |
| 2019 | **Hide and Seek** (OpenAI) | Агенты в прятках сами изобретают использование инструментов: строят укрытия, «сёрфят» на ящиках. | [блог с гифками](https://openai.com/index/emergent-tool-use/) |

Почему игры? Есть готовый симулятор, чёткая награда (счёт, победа) и можно сыграть миллионы партий. Это идеальный полигон.

### Робототехника

* [Роботы-футболисты DeepMind](https://sites.google.com/view/op3-soccer): походка, удары, подъём после падения — всё выучено в симуляторе и перенесено на железо (sim-to-real).
* [Обучение ходьбе ANYmal](https://arxiv.org/abs/1901.08652) и [«научиться ходить за минуты»](https://arxiv.org/abs/2109.11978): RL-контроллеры для четвероногих роботов, которые устойчивее ручных.
* [Boston Dynamics: RL для Spot](https://bostondynamics.com/blog/starting-on-the-right-foot-with-reinforcement-learning/): коммерческий робот, часть контроллеров которого теперь обучается, а не программируется.
* [Кубик Рубика одной рукой](https://openai.com/index/solving-rubiks-cube/): пример, насколько трудно перенести политику из симуляции в реальный мир.

### Управление инфраструктурой

* [Охлаждение дата-центров Google](https://deepmind.google/discover/blog/deepmind-ai-reduces-google-data-centre-cooling-bill-by-40/): агент управляет насосами, чиллерами и вентиляцией; минус 40% энергии на охлаждение. Состояние — сотни датчиков, действие — уставки оборудования, награда — энергия при соблюдении температурных ограничений.
* [Управление плазмой в токамаке](https://www.nature.com/articles/s41586-021-04301-9) (DeepMind и EPFL, 2022): RL-контроллер держит форму плазмы, меняя токи в 19 магнитных катушках.
* Управление светофорами, балансировка нагрузки в сетях, планирование размещения блоков на чипе ([AlphaChip](https://deepmind.google/discover/blog/how-alphachip-transformed-computer-chip-design/)).

### Финансы: торговля на бирже

Естественная постановка: состояние — цены, объёмы, позиция; действие — купить, продать, держать; награда — прибыль с учётом риска. Реально применяется для **исполнения крупных ордеров** (как разбить заявку на части, чтобы не сдвинуть цену) и **маркет-мейкинга** (какие котировки выставлять). Честная оговорка: рынок нестационарен (правила меняются, когда все начинают действовать одинаково), сигнал очень шумный, а ошибки стоят денег, поэтому «обучить агента, который торгует лучше всех» — гораздо сложнее, чем обыграть Atari. Обзор: [Deep RL for trading](https://arxiv.org/abs/1911.10107), учебная библиотека [FinRL](https://github.com/AI4Finance-Foundation/FinRL).

### Роевое поведение дронов

Несколько агентов учатся одновременно: держать строй, облетать препятствия, вместе искать цель, делить между собой задачи. Каждый дрон видит только соседей, а награда общая на всех — это **multi-agent RL**, неделя 12. Примеры: [обучение рою квадрокоптеров летать сквозь лес](https://arxiv.org/abs/2202.03308), [RL для координации дронов в симуляторе](https://arxiv.org/abs/2103.10412); для экспериментов есть среды [PettingZoo](https://pettingzoo.farama.org/) и [gym-pybullet-drones](https://github.com/utiasDSL/gym-pybullet-drones).

### Рекомендации и реклама

Что показать пользователю, чтобы он остался надолго, а не только кликнул сейчас: награда — долгосрочная вовлечённость, действие — выбор контента. Плюс A/B-тесты и подбор баннеров, то есть те самые бандиты из раздела 7.

### Языковые модели

* **RLHF** ([InstructGPT, 2022](https://arxiv.org/abs/2203.02155)): ChatGPT стал «полезным собеседником» именно благодаря RL на человеческих предпочтениях. Состояние — диалог, действие — следующее слово, награда — оценка людей. Разберём на неделе 14.
* **Рассуждающие модели** ([DeepSeek-R1, 2025](https://arxiv.org/abs/2501.12948)): RL на проверяемых наградах (правильно ли решена задача) учит модель длинным цепочкам рассуждений.

### Наука

[AlphaTensor](https://deepmind.google/discover/blog/discovering-novel-algorithms-with-alphatensor/) и [AlphaDev](https://deepmind.google/discover/blog/alphadev-discovers-faster-sorting-algorithms/): поиск алгоритмов умножения матриц и сортировки как игра, где ход — шаг алгоритма, а награда — его скорость.

Третий итог лекции: **Reinforcement Learning широко применяется в самых разных областях: от игр до управления охлаждением дата-центров и от торговли на бирже до роевого поведения дронов.**

### Интерактивные демо для домашнего «потыкать»

* [ReinforceJS](https://cs.stanford.edu/people/karpathy/reinforcejs/) (A. Karpathy): GridWorld с обучением прямо в браузере, PuckWorld, WaterWorld. Отличная иллюстрация к неделям 2–5.
* [Каталог сред Gymnasium](https://gymnasium.farama.org/environments/classic_control/): гифки всех стандартных сред, от CartPole до Atari и MuJoCo.
* [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction): бесплатный курс с ноутбуками, можно обучить и выложить своего агента.
* [OpenAI Spinning Up](https://spinningup.openai.com/): короткие и чистые реализации основных алгоритмов + список ключевых статей.

## 10. Карта курса

![taxonomy](../../assets/rl_taxonomy.png)

Первые четыре недели — **табличные методы**: состояний мало, всё можно хранить в массиве, и на них видно устройство алгоритмов без шума нейросетей. Дальше те же идеи переносим на нейросети (**deep RL**), потом расширяем: обучение модели среды, exploration, обучение по логам без взаимодействия, несколько агентов, RL для языковых моделей.

![course](../../assets/course_map.png)

Всё, что в этой таблице, мы **реализуем сами** с нуля: от простейших бандитов до PPO и SAC. Это принципиальная позиция курса: RL-алгоритмы печально известны тем, что «почти работающая» реализация не работает вовсе, и понять, где ошибка, можно только зная каждую строчку.

## Итоги лекции

1. **Reinforcement Learning зарождался в психологии.** Закон эффекта Торндайка и «подкрепление» Скиннера — это описание того же механизма, который мы теперь программируем; математику к нему добавила теория управления.
2. **Reinforcement Learning можно выделить в отдельную группу алгоритмов машинного обучения.** Нет правильных ответов, награда отложена, данные порождает сам агент, нужно исследовать. Ни одна из этих черт не встречается в supervised и unsupervised learning.
3. **Reinforcement Learning широко применяется в самых разных областях: от игр до управления охлаждением дата-центров и от торговли на бирже до роевого поведения дронов.** Рецепт один: среда + числовая цель.

Словарь, который нужно унести с собой: **агент, среда, состояние (наблюдение), действие, награда, политика, эпизод, суммарная награда, exploration и exploitation**.

**Что дальше.** Неделя 2: многорукие бандиты (та самая задача с кафе, но с настоящими алгоритмами), марковские процессы принятия решений и уравнения Беллмана — математический фундамент всего курса.

## Литература и ссылки

**Базовые**

* R. Sutton, A. Barto. *Reinforcement Learning: An Introduction*, 2nd ed. — [бесплатный PDF](http://incompleteideas.net/book/the-book-2nd.html). Глава 1 покрывает сегодняшнюю лекцию; раздел 1.7 «Early History of Reinforcement Learning» — про психологические корни подробно.
* D. Silver. [UCL Course on RL](https://www.davidsilver.uk/teaching/) — 10 лекций с видео, классика. Лекция 1 — введение.
* S. Levine. [CS285: Deep RL (Berkeley)](https://rail.eecs.berkeley.edu/deeprlcourse/) — про deep RL, пригодится с недели 5.
* [OpenAI Spinning Up](https://spinningup.openai.com/) — короткое введение + чистые реализации.

**Практика**

* [Gymnasium](https://gymnasium.farama.org/) — документация по средам и интерфейсу.
* [CleanRL](https://github.com/vwxyzjn/cleanrl) — каждый алгоритм в одном файле, удобно читать.
* [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) — библиотека для быстрых экспериментов и сравнения со своей реализацией.
* [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction).

**Статьи, упомянутые сегодня**

* Mnih et al. [Human-level control through deep RL](https://www.nature.com/articles/nature14236) (DQN), 2015.
* Silver et al. [Mastering the game of Go without human knowledge](https://www.nature.com/articles/nature24270) (AlphaGo Zero), 2017.
* Degrave et al. [Magnetic control of tokamak plasmas through deep RL](https://www.nature.com/articles/s41586-021-04301-9), 2022.
* Ouyang et al. [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155) (InstructGPT), 2022.
* DeepSeek-AI. [DeepSeek-R1](https://arxiv.org/abs/2501.12948), 2025.

## На семинаре

* Интерфейс Gymnasium: `reset`, `step`, `action_space`, `observation_space` на FrozenLake и CartPole (см. `../seminar/seminar.ipynb`).
* Пишем политики руками: улучшаем эвристику для CartPole, составляем таблицу-политику для FrozenLake, смотрим, что происходит на «скользком» льду.
* **Мини-семинар по PyTorch** для тех, кто с ним не работал: `../seminar/pytorch_intro.ipynb`. Тензоры, autograd, `nn.Module`, цикл обучения и первый «агент на нейросети» для CartPole.

## Домашнее задание

См. `../homework/homework.ipynb`: ручные политики для CartPole и FrozenLake, формулировка трёх задач из жизни на языке RL (агент, среда, состояние, действие, награда), мини-эксперимент с исследованием.